In [4]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Ready for training!")

PyTorch version: 2.12.1+cpu
Torchvision version: 0.27.1+cpu
Ready for training!


In [5]:
# Transforms = what to do to each image before feeding to model
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),      # Resize to 224x224
    transforms.RandomHorizontalFlip(),  # Randomly flip image (data augmentation)
    transforms.RandomRotation(10),      # Randomly rotate by 10 degrees
    transforms.ToTensor(),              # Convert to PyTorch tensor
    transforms.Normalize(               # Normalize with ImageNet values
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),      # Only resize and normalize for val
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms ready!")
print("Train transforms:", len(train_transforms.transforms), "steps")
print("Val transforms  :", len(val_transforms.transforms), "steps")

Transforms ready!
Train transforms: 5 steps
Val transforms  : 3 steps


In [6]:
# Load datasets from our train and val folders
train_dataset = datasets.ImageFolder("data/train", transform=train_transforms)
val_dataset   = datasets.ImageFolder("data/val",   transform=val_transforms)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

# Check class names
print("Classes:", train_dataset.classes)
print()
print("Train images:", len(train_dataset))
print("Val images  :", len(val_dataset))
print()
print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))

Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

Train images: 2019
Val images  : 508

Train batches: 64
Val batches  : 16


In [7]:
import torch.nn as nn
from torchvision import models

# Load pretrained EfficientNet-B0
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# Freeze all layers - we don't want to change what it already knows
for param in model.parameters():
    param.requires_grad = False

# Replace the last layer with our own - 6 classes instead of 1000
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(1280, 6)  # 1280 inputs → 6 waste categories
)

# Count trainable parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model: EfficientNet-B0")
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters   : {total - trainable:,}")
print()
print("Only training the last layer - transfer learning!")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\anjan/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:01<00:00, 18.7MB/s]


Model: EfficientNet-B0
Total parameters    : 4,015,234
Trainable parameters: 7,686
Frozen parameters   : 4,007,548

Only training the last layer - transfer learning!


In [8]:
import torch.optim as optim

# Loss function - measures how wrong the model is
criterion = nn.CrossEntropyLoss()

# Optimizer - adjusts weights to reduce the loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function : CrossEntropyLoss")
print("Optimizer     : Adam")
print("Learning rate : 0.001")
print()
print("Ready to train!")

Loss function : CrossEntropyLoss
Optimizer     : Adam
Learning rate : 0.001

Ready to train!


In [9]:
# Training loop
num_epochs = 5
device = torch.device("cpu")
model = model.to(device)

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0
    train_correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()          # Clear previous gradients
        outputs = model(images)        # Forward pass
        loss = criterion(outputs, labels)  # Calculate loss
        loss.backward()                # Backward pass
        optimizer.step()               # Update weights

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_correct += predicted.eq(labels).sum().item()

    # Validation phase
    model.eval()
    val_loss = 0
    val_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()

    # Print results
    train_acc = 100 * train_correct / len(train_dataset)
    val_acc   = 100 * val_correct / len(val_dataset)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.3f} | "
          f"Train Acc: {train_acc:.1f}% | "
          f"Val Loss: {val_loss/len(val_loader):.3f} | "
          f"Val Acc: {val_acc:.1f}%")

print("\nTraining complete!")

Epoch 1/5 | Train Loss: 1.266 | Train Acc: 59.7% | Val Loss: 0.918 | Val Acc: 76.0%
Epoch 2/5 | Train Loss: 0.847 | Train Acc: 73.9% | Val Loss: 0.732 | Val Acc: 79.1%
Epoch 3/5 | Train Loss: 0.721 | Train Acc: 77.3% | Val Loss: 0.657 | Val Acc: 79.1%
Epoch 4/5 | Train Loss: 0.649 | Train Acc: 78.7% | Val Loss: 0.627 | Val Acc: 80.5%
Epoch 5/5 | Train Loss: 0.608 | Train Acc: 80.1% | Val Loss: 0.613 | Val Acc: 78.9%

Training complete!
